# module-composition — ex2: rebuild the MLP with nn.Sequential

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-composition`. Running the final beacon cell reports progress against the `PyTorch: Module composition` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Module composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-composition`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-composition"
DD_SUBTOPIC = "PyTorch: Module composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Module composition — quick refresher

Modules compose by assignment. Any `nn.Module` instance assigned to `self.<name>` inside `__init__` is auto-registered as a child module — visible in `.children()`, `.named_modules()`, and recursively included in `.parameters()` / `.state_dict()`.

**Two composition styles.**
1. **Named attributes** — `self.linear1 = nn.Linear(...)`, `self.linear2 = nn.Linear(...)`. Use when the forward pass branches (residual blocks, attention heads, gating).
2. **`nn.Sequential(*modules)`** — wraps a list of Modules into a single callable that pipes the output of one into the input of the next. Use when the forward is a strict left-to-right pipeline.

**Lists need `nn.ModuleList`, not `list`.** A plain Python list of Modules assigned to an attribute does NOT register the children — their parameters become invisible. Use `nn.ModuleList(...)` (registers + supports indexing) or `nn.Sequential(...)` (registers + auto-pipes).

### Exercise 2 — rebuild the MLP with nn.Sequential

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Build the same two-layer MLP using nn.Sequential as a single self.net attribute, and confirm forward-pass parity with the named-attribute version from the previous drill.
> Keywords: nn.Sequential, pipeline, composition, state_dict-parity
> ```

**KCs targeted:** `sequential-pipeline`, `child-module-attribute-registration`

Implement `SequentialMLP` — the exact same two-layer MLP as the previous exercise but composed via `nn.Sequential` instead of named attributes. In `__init__(self, in_features, hidden_features, out_features)`:

1. Call `super().__init__()`.
2. Build `self.net = t.nn.Sequential(linear1, relu, linear2)` where:
   - `linear1 = t.nn.Linear(in_features, hidden_features)`
   - `relu = t.nn.ReLU()`
   - `linear2 = t.nn.Linear(hidden_features, out_features)`
3. `forward(self, x: Tensor) -> Tensor` just returns `self.net(x)`.

Return an instance from `ex2_build_sequential_mlp(in_features, hidden_features, out_features)`.

The big idea: `Sequential` is itself a Module, so the whole pipeline lives behind a single child attribute named `net`. Compared to the named-attribute version (`fc1` + `fc2`), the names of the children change (`net.0.weight` not `fc1.weight`) but the parameter count and forward output match exactly.

The test asserts: (a) `self.net` is an `nn.Sequential` of length 3, (b) parameter names are now `net.0.*` / `net.2.*`, (c) forward output matches the equivalent named-attribute MLP when their parameters are copied across.

In [ ]:
def ex2_build_sequential_mlp(in_features: int, hidden_features: int, out_features: int):
    """Return a SequentialMLP instance — same forward as TwoLayerMLP but composed via nn.Sequential."""
    raise NotImplementedError()


def _test_ex2():
    mod = ex2_build_sequential_mlp(in_features=4, hidden_features=8, out_features=3)
    assert isinstance(mod, t.nn.Module)

    # Top-level child should be exactly one — the Sequential named 'net'.
    named_children = dict(mod.named_children())
    assert set(named_children.keys()) == {'net'}, (
        f'expected exactly one top-level child named "net", got {set(named_children.keys())}'
    )
    assert isinstance(named_children['net'], t.nn.Sequential), (
        f'expected nn.Sequential, got {type(named_children["net"]).__name__}'
    )
    assert len(named_children['net']) == 3, (
        f'Sequential should contain 3 modules (Linear, ReLU, Linear), got {len(named_children["net"])}'
    )

    # Element types inside the Sequential.
    seq = named_children['net']
    assert isinstance(seq[0], t.nn.Linear)
    assert isinstance(seq[1], t.nn.ReLU)
    assert isinstance(seq[2], t.nn.Linear)
    assert seq[0].in_features == 4 and seq[0].out_features == 8
    assert seq[2].in_features == 8 and seq[2].out_features == 3

    # Param names: Sequential children are auto-named by integer index → 'net.0.weight' etc.
    named_params = dict(mod.named_parameters())
    expected_names = {'net.0.weight', 'net.0.bias', 'net.2.weight', 'net.2.bias'}
    assert set(named_params.keys()) == expected_names, (
        f'expected {expected_names}, got {set(named_params.keys())}'
    )

    # Forward shape check.
    x = t.randn(5, 4, generator=t.Generator().manual_seed(7))
    y = mod(x)
    assert y.shape == (5, 3), f'expected (5, 3), got {tuple(y.shape)}'

    # Parity check: build a reference Sequential by hand and copy params over,
    # then verify outputs agree.
    reference = t.nn.Sequential(
        t.nn.Linear(4, 8),
        t.nn.ReLU(),
        t.nn.Linear(8, 3),
    )
    with t.no_grad():
        reference[0].weight.copy_(seq[0].weight)
        reference[0].bias.copy_(seq[0].bias)
        reference[2].weight.copy_(seq[2].weight)
        reference[2].bias.copy_(seq[2].bias)
    y_ref = reference(x)
    assert t.allclose(y, y_ref, atol=1e-6), (
        f'forward mismatch vs reference Sequential — your forward should be self.net(x)'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_build_sequential_mlp(in_features: int, hidden_features: int, out_features: int):
    class SequentialMLP(t.nn.Module):
        def __init__(self, in_features, hidden_features, out_features):
            super().__init__()
            self.net = t.nn.Sequential(
                t.nn.Linear(in_features, hidden_features),
                t.nn.ReLU(),
                t.nn.Linear(hidden_features, out_features),
            )
        def forward(self, x: Tensor) -> Tensor:
            return self.net(x)
    return SequentialMLP(in_features, hidden_features, out_features)
```

**Sequential is just a Module that auto-pipes.** `Sequential(A, B, C)(x)` is exactly `C(B(A(x)))`. It stores its children in `_modules` keyed by string index ('0', '1', '2'), which is why `named_parameters()` produces `net.0.weight` etc.

**When to use Sequential vs named attributes.** Sequential is cleanest when (a) the forward is a strict left-to-right pipeline and (b) you never want to introspect a specific child by a meaningful name. The moment your forward branches (residual block, attention, gating) or you want `self.encoder` / `self.decoder` to be separately introspectable, switch back to named attributes.

**ARENA's actual `make_cnn`** uses Sequential as the outer container with a long Conv2d → BN → ReLU → ... stack. Knowing this composition style is non-negotiable for that exercise.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()